In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
import torch
from peft import PeftModel

c:\imar6teen\projects\NotebookLM Clone\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1006 00:08:55.435000 22768 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


In [ ]:
FINE_TUNED_MODEL_DIR = "./qlora_gemma_4_e4b/checkpoint-450"
BASE_MODEL = "google/gemma-4-e4b-it"

processor = AutoProcessor.from_pretrained(BASE_MODEL)
base_model = AutoModelForMultimodalLM.from_pretrained(BASE_MODEL)

Loading weights: 100%|██████████| 2076/2076 [00:51<00:00, 40.39it/s] 


In [ ]:
base_model.get_memory_footprint() / 1e6 # the memory increases due to adding custom layers which are LoRA

15882.207856

In [ ]:
del base_model

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit= True,
    bnb_4bit_quant_type= "nf4",
    bnb_4bit_use_double_quant= True,
    bnb_4bit_compute_dtype= torch.float32,
)

model = AutoModelForMultimodalLM.from_pretrained(FINE_TUNED_MODEL_DIR, quantization_config= bnb_config)
model = PeftModel.from_pretrained(model, FINE_TUNED_MODEL_DIR)

Loading weights: 100%|██████████| 516/516 [00:00<00:00, 880.00it/s] 
c:\imar6teen\projects\NotebookLM Clone\backend\.venv\Lib\site-packages\peft\tuners\tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
model.get_memory_footprint() / 1e6

9299.640944

# Example

In [ ]:
message1 = [
    {'role': 'user', 'content': 'The birch canoe slid on the smooth planks.'}
]

message2 = [
    {'role' : "system", "content" : "you are helpful assistant."},
    {"role" : "user", "content" : "who are you?"}
]

messages = [message1, message2]

In [ ]:
inputs = processor.apply_chat_template(
    messages,
    enable_thinking=False,
    return_dict=True,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

In [ ]:
inputs

{'input_ids': tensor([[     0,      0,      0,      0,      0,      2,    105,   2364,    107,
            818,  80470,  62248,  76068,    580,    506,   7080, 100944, 236761,
            106,    107,    105,   4368,    107],
        [     2,    105,   9731,    107,   7624,    659,  11045,  16326, 236761,
            106,    107,    105,   2364,    107,  14625,    659,    611, 236881,
            106,    107,    105,   4368,    107]], device='cuda:0'), 'attention_mask': tensor([[0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0'), 'mm_token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],
       device='cuda:0')}

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=512
)

In [ ]:
processor.decode(outputs[:, input_len:], skip_special_tokens=True)

['On the smooth planks, the birch canoe slid. Yes, hrrmmm.',
 'Helpful assistant, you are to whom?']

# Save

In [ ]:
model = model.merge_and_unload()

c:\imar6teen\projects\NotebookLM Clone\backend\.venv\Lib\site-packages\peft\tuners\lora\bnb.py:377: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [ ]:
model._hf_peft_config_loaded = False
processor.save_pretrained("./qlora_gemma_4_e4b_model")
model.save_pretrained("./qlora_gemma_4_e4b_model")

Writing model shards: 100%|██████████| 1/1 [00:45<00:00, 45.85s/it]


In [ ]:
import os


repo_id = "imar6teen/gemma_4_e4b_model_quantized"
model.push_to_hub(repo_id, private=False)
processor.push_to_hub(repo_id, private=False)

Writing model shards:   0%|          | 0/1 [00:08<?, ?it/s]


KeyboardInterrupt: 